# Herramienta 03 — Recomendación personalizada de destinos de viaje

Sistema híbrido que combina **filtrado colaborativo** (usuarios similares) y **filtrado por contenido** (preferencias declaradas) para sugerir destinos dentro de las rutas de la empresa.

**Dataset:** [Travel Recommendation Dataset](https://www.kaggle.com/datasets/amanmehra23/travel-recommendation-dataset) — 1 000 destinos, 999 usuarios, ~2 000 interacciones (reviews + historial de viajes).

**Entregables:**
- Sistema híbrido CF + contenido
- Métricas: Precision@5, Recall@5, NDCG@5
- Ejemplos para múltiples usuarios
- Análisis de efectividad (satisfacción, demanda proyectada, cobertura)

## 1. Configuración

Se usan `pandas`, `numpy` y `scikit-learn`. El notebook carga los datos directamente desde el repositorio — no requiere instalar nada adicional en Colab.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

SEED  = 42
K     = 5      # Top-K recomendaciones
ALPHA = 0.7    # Peso del filtrado colaborativo vs. contenido
np.random.seed(SEED)

BASE = 'https://raw.githubusercontent.com/AndresGuido9820/sistema-transporte-inteligente/main/data/processed'
print('Configuración lista. Alpha CF =', ALPHA, '| Alpha contenido =', round(1 - ALPHA, 1))

## 2. Carga de datos

El dataset proviene de Kaggle ([amanmehra23/travel-recommendation-dataset](https://www.kaggle.com/datasets/amanmehra23/travel-recommendation-dataset)) y contiene cuatro tablas:

| Tabla | Filas | Descripción |
|-------|-------|-------------|
| `destinations.csv` | 1 000 | Destinos con nombre, estado, tipo y popularidad |
| `users.csv` | 999 | Perfil del usuario: preferencias declaradas, grupo familiar |
| `reviews.csv` | 999 | Calificaciones usuario → destino (Rating 1-5) |
| `user_history.csv` | 999 | Visitas históricas con ExperienceRating (1-5) |

Reviews e historial se combinan en una sola tabla de interacciones promediando ratings duplicados.

In [ ]:
destinations = pd.read_csv(f'{BASE}/destinations.csv')
users        = pd.read_csv(f'{BASE}/users.csv')
reviews      = pd.read_csv(f'{BASE}/reviews.csv')
history      = pd.read_csv(f'{BASE}/user_history.csv')

print(f'Destinos  : {len(destinations):,} | Tipos: {sorted(destinations.Type.unique())}')
print(f'Usuarios  : {len(users):,}')
print(f'Reviews   : {len(reviews):,} | Rating range: {reviews.Rating.min()}–{reviews.Rating.max()}')
print(f'Historial : {len(history):,} | Rating range: {history.ExperienceRating.min()}–{history.ExperienceRating.max()}')

# Combinar reviews + historial
rev_c  = reviews[['UserID','DestinationID','Rating']].rename(columns={'Rating':'rating'})
hist_c = history[['UserID','DestinationID','ExperienceRating']].rename(columns={'ExperienceRating':'rating'})
interactions = (pd.concat([rev_c, hist_c])
                  .groupby(['UserID','DestinationID'])['rating']
                  .mean()
                  .reset_index())

print(f'\nInteracciones combinadas: {len(interactions):,}')
print(f'Usuarios únicos : {interactions.UserID.nunique():,}')
print(f'Destinos únicos : {interactions.DestinationID.nunique():,}')
interactions.head()

## 3. Análisis exploratorio

Se examina la distribución de tipos de destino, calificaciones y preferencias declaradas de los usuarios, más los 10 destinos con mayor popularidad registrada.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Tipos de destino
tc = destinations['Type'].value_counts()
axes[0].bar(tc.index, tc.values, color=['#2f6fff','#11a87d','#f6a531','#e06d2f','#8a5cf5'])
axes[0].set_title('Tipos de destino', fontweight='bold')
axes[0].set_ylabel('Cantidad')
axes[0].grid(axis='y', alpha=0.25)

# Distribución de calificaciones
interactions['rating'].round().astype(int).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='#2f6fff')
axes[1].set_title('Distribución de calificaciones', fontweight='bold')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Interacciones')
axes[1].grid(axis='y', alpha=0.25)
axes[1].tick_params(axis='x', rotation=0)

# Preferencias declaradas
PMAP = {'Beaches':'Beach','Historical':'Historical','Nature':'Nature',
        'Adventure':'Adventure','City':'City'}
all_prefs = []
for p in users['Preferences'].dropna():
    all_prefs.extend([PMAP.get(x.strip(), x.strip()) for x in p.split(',')])
pd.Series(all_prefs).value_counts().plot(kind='bar', ax=axes[2], color='#11a87d')
axes[2].set_title('Preferencias declaradas por usuarios', fontweight='bold')
axes[2].set_ylabel('Usuarios')
axes[2].grid(axis='y', alpha=0.25)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('Top 10 destinos por popularidad:')
display(destinations.sort_values('Popularity', ascending=False)[
    ['Name','State','Type','Popularity']].head(10).reset_index(drop=True).round(2))

## 4. Construcción del sistema híbrido

El sistema combina dos señales complementarias:

| Componente | Señal usada | Pregunta que responde |
|------------|-------------|----------------------|
| **Filtrado colaborativo (CF)** | Matriz de ratings usuario × destino | ¿Qué les gustó a usuarios similares a mí? |
| **Filtrado por contenido (CB)** | Preferences declaradas + Destination.Type + Popularity | ¿Qué destinos coinciden con mis preferencias? |
| **Híbrido** | CF × 0.7 + CB × 0.3 | Balance entre historial colectivo y perfil individual |

### 4.1 Filtrado colaborativo (User-User CF)

Se construye la matriz usuario-destino, se calcula similitud coseno entre usuarios (diagonal en 0 para evitar auto-similitud) y los scores CF son el promedio ponderado de ratings de usuarios similares. Se normalizan a [0, 1] por usuario.

In [ ]:
# Matriz usuario-destino
matrix = interactions.pivot_table(
    index='UserID', columns='DestinationID', values='rating', fill_value=0)
print(f'Matriz usuario-destino: {matrix.shape[0]} usuarios × {matrix.shape[1]} destinos')
print(f'Densidad: {(matrix > 0).values.mean():.2%}')

# Similitud coseno (sin auto-similitud)
sim_arr = cosine_similarity(matrix.values)
np.fill_diagonal(sim_arr, 0)

# Scores CF: suma ponderada de ratings de usuarios similares
cf_raw = pd.DataFrame(
    sim_arr @ matrix.values,
    index=matrix.index, columns=matrix.columns)

# Normalizar por usuario a [0, 1]
cf_min = cf_raw.min(axis=1)
cf_max = cf_raw.max(axis=1)
cf_norm = cf_raw.sub(cf_min, axis=0).div((cf_max - cf_min).replace(0, 1), axis=0)

print('\nScores CF calculados y normalizados.')
print(f'Usuarios con ≥ 3 interacciones: {(matrix > 0).sum(axis=1).ge(3).sum()}')

### 4.2 Filtrado por contenido

Las preferencias declaradas del usuario (ej. `"Beaches, Historical"`) se mapean al tipo de destino (`Beach`, `Historical`). El score de contenido vale **0.7** si hay coincidencia de tipo, más un **0.3** de bonus proporcional a la popularidad del destino.

In [ ]:
users_idx = users.set_index('UserID').copy()
users_idx['pref_set'] = users_idx['Preferences'].apply(
    lambda p: {PMAP.get(x.strip(), x.strip()) for x in p.split(',')}
    if pd.notna(p) else set())

dest_type = destinations.set_index('DestinationID')['Type']
dest_pop  = destinations.set_index('DestinationID')['Popularity']
pop_norm  = (dest_pop - dest_pop.min()) / (dest_pop.max() - dest_pop.min())

cb_data = {}
for uid in matrix.index:
    prefs = users_idx.loc[uid, 'pref_set'] if uid in users_idx.index else set()
    cb_data[uid] = {
        did: 0.7 * (1.0 if dest_type.get(did) in prefs else 0.0)
             + 0.3 * float(pop_norm.get(did, 0.5))
        for did in matrix.columns
    }

cb_norm = (pd.DataFrame(cb_data).T
             .reindex(index=matrix.index, columns=matrix.columns)
             .fillna(0))

print('Scores de contenido calculados.')
print('Usuarios con preferencias conocidas:', users_idx['pref_set'].apply(bool).sum())

### 4.3 Combinación híbrida y función de recomendación

`score_hibrido = 0.7 × score_CF + 0.3 × score_contenido`

Los destinos ya visitados por el usuario se excluyen del ranking. La justificación de cada recomendación indica si hubo coincidencia de tipo o si proviene puramente de usuarios similares.

In [ ]:
hybrid = ALPHA * cf_norm + (1 - ALPHA) * cb_norm

def recomendar(user_id, k=K, mode='hybrid'):
    """Retorna DataFrame con Top-K recomendaciones para user_id."""
    if user_id not in matrix.index:
        return pd.DataFrame()
    seen   = set(matrix.columns[matrix.loc[user_id] > 0])
    scores = {'cf': cf_norm, 'content': cb_norm, 'hybrid': hybrid}[mode].loc[user_id]
    cands  = scores[~scores.index.isin(seen)].sort_values(ascending=False).head(k)
    prefs  = users_idx.loc[user_id, 'pref_set'] if user_id in users_idx.index else set()

    rows = []
    for rank, (did, score) in enumerate(cands.items(), 1):
        dtype  = dest_type.get(did, '?')
        name_r = destinations.loc[destinations.DestinationID == did, 'Name'].values
        state_r= destinations.loc[destinations.DestinationID == did, 'State'].values
        reason = ('Coincide con tu preferencia + usuarios similares'
                  if dtype in prefs else 'Altamente valorado por usuarios con perfil similar')
        rows.append({'Rank': rank,
                     'Destino': name_r[0] if len(name_r) else f'Dest_{did}',
                     'Tipo': dtype,
                     'Estado': state_r[0] if len(state_r) else '?',
                     'Score': round(float(score), 3),
                     'Razón': reason})
    return pd.DataFrame(rows)

print(f'Sistema híbrido listo. Alpha CF = {ALPHA} | Alpha contenido = {1 - ALPHA}')
print(f'Forma de la matriz de scores: {hybrid.shape}')

## 5. Evaluación Leave-One-Out

Para cada usuario con al menos 2 destinos con rating ≥ 4, se oculta uno al azar y se verifica si el recomendador lo recupera en el Top-K. Se comparan los tres modos.

- **Precision@K** = hits / (usuarios evaluados × K)
- **Recall@K** = proporción de usuarios cuyo item oculto aparece en Top-K (equivalente al Hit Rate con 1 item relevante)
- **NDCG@K** = penaliza las recomendaciones correctas que aparecen en posiciones bajas

In [ ]:
def evaluar_luo(mode='hybrid', k=K):
    rng = np.random.default_rng(SEED)
    resultados = []
    scores_df = {'cf': cf_norm, 'content': cb_norm, 'hybrid': hybrid}[mode]

    for uid in matrix.index:
        rated_high = matrix.columns[matrix.loc[uid] >= 4].tolist()
        if len(rated_high) < 2:
            continue
        hidden = rng.choice(rated_high)
        seen   = set(matrix.columns[matrix.loc[uid] > 0]) - {hidden}
        s      = scores_df.loc[uid]
        top_k  = s[~s.index.isin(seen)].sort_values(ascending=False).head(k).index.tolist()
        hit    = int(hidden in top_k)
        ndcg   = (1.0 / np.log2(top_k.index(hidden) + 2)) if hit else 0.0
        resultados.append({'hit': hit, 'ndcg': ndcg})

    df = pd.DataFrame(resultados)
    return {'Precision@K': round(df.hit.mean() / k, 4),
            'Recall@K':    round(df.hit.mean(), 4),
            'NDCG@K':      round(df.ndcg.mean(), 4),
            'Hit Rate':    round(df.hit.mean(), 4),
            'N evaluados': len(df)}

print('Evaluando...')
r_cf  = evaluar_luo('cf')
r_cb  = evaluar_luo('content')
r_hy  = evaluar_luo('hybrid')

eval_df = pd.DataFrame([r_cf, r_cb, r_hy], index=['Colaborativo', 'Contenido', 'Híbrido'])
display(eval_df)

fig, ax = plt.subplots(figsize=(8, 3))
x = range(3)
modelos = ['Colaborativo', 'Contenido', 'Híbrido']
hits    = [r_cf['Hit Rate'], r_cb['Hit Rate'], r_hy['Hit Rate']]
bars = ax.bar(modelos, hits, color=['#2f6fff','#f6a531','#11a87d'])
for bar, val in zip(bars, hits):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', fontweight='bold')
ax.set_title('Hit Rate por modo (Leave-One-Out, K=5)', fontweight='bold')
ax.set_ylabel('Hit Rate')
ax.set_ylim(0, max(hits) * 1.25)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Ejemplos para múltiples usuarios

Se muestran recomendaciones para 6 usuarios con perfiles distintos para ilustrar la personalización del sistema.

In [ ]:
# Seleccionar 6 usuarios con perfiles variados
sample_uids = []
seen_prefs = set()
for uid in matrix.index:
    if uid not in users_idx.index:
        continue
    prefs_key = frozenset(users_idx.loc[uid, 'pref_set'])
    if prefs_key not in seen_prefs:
        sample_uids.append(uid)
        seen_prefs.add(prefs_key)
    if len(sample_uids) == 6:
        break

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, uid in enumerate(sample_uids):
    recs  = recomendar(uid, k=5)
    prefs = ', '.join(sorted(users_idx.loc[uid, 'pref_set'])) if uid in users_idx.index else 'N/A'
    nombre= users_idx.loc[uid, 'Name'] if uid in users_idx.index else str(uid)
    axes[i].barh(recs['Destino'][::-1], recs['Score'][::-1], color='#2f6fff', alpha=0.85)
    axes[i].set_title(f'{nombre} (ID {uid})\nPreferencias: {prefs}', fontsize=9)
    axes[i].set_xlabel('Score híbrido')
    axes[i].tick_params(axis='y', labelsize=7)
    axes[i].grid(axis='x', alpha=0.25)

plt.suptitle('Top-5 recomendaciones para 6 perfiles distintos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla detallada
print('\nTabla de recomendaciones con justificación:')
for uid in sample_uids[:3]:
    prefs  = ', '.join(sorted(users_idx.loc[uid, 'pref_set'])) if uid in users_idx.index else 'N/A'
    nombre = users_idx.loc[uid, 'Name'] if uid in users_idx.index else str(uid)
    recs   = recomendar(uid, k=3)
    print(f'\n{nombre} (ID {uid}) | Preferencias: {prefs}')
    display(recs)

## 7. Herramienta interactiva

Seleccione un usuario y el modo de recomendación. Se muestra el perfil, el historial de viajes y las recomendaciones generadas con su justificación.

In [ ]:
#@title Parámetros de la herramienta
usuario_id           = 1     #@param {type:'integer'}
top_k                = 5     #@param {type:'integer'}
modo_recomendacion   = 'hybrid' #@param ['hybrid', 'cf', 'content']

if usuario_id not in matrix.index:
    usuario_id = int(matrix.index[0])
    print(f'Usuario no encontrado. Usando: {usuario_id}')

# Perfil
if usuario_id in users_idx.index:
    u = users_idx.loc[usuario_id]
    print(f'Usuario  : {u["Name"]} (ID {usuario_id})')
    print(f'Género   : {u["Gender"]} | Adultos: {u["NumberOfAdults"]} | Niños: {u["NumberOfChildren"]}')
    print(f'Preferencias declaradas: {u["Preferences"]}')

# Historial
hist_u = (interactions[interactions.UserID == usuario_id]
          .merge(destinations[['DestinationID','Name','Type','State']], on='DestinationID', how='left')
          .sort_values('rating', ascending=False))
print(f'\nHistorial de interacciones ({len(hist_u)} destinos visitados):')
display(hist_u[['Name','Type','State','rating']].head(8).reset_index(drop=True))

# Recomendaciones
recs = recomendar(usuario_id, k=top_k, mode=modo_recomendacion)
print(f'\nTop {top_k} recomendaciones (modo: {modo_recomendacion}):')
display(recs)

plt.figure(figsize=(9, 3))
plt.barh(recs['Destino'][::-1], recs['Score'][::-1], color='#2f6fff', alpha=0.85)
plt.xlabel('Score')
plt.title(f'Top {top_k} para Usuario {usuario_id} (modo: {modo_recomendacion})', fontweight='bold')
plt.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Análisis de efectividad

Se evalúan tres dimensiones de efectividad:

1. **Usuarios satisfechos:** Hit Rate por modo — qué porcentaje de usuarios recibe al menos una recomendación relevante en Top-5.
2. **Incremento en demanda de rutas:** Destinos más frecuentemente recomendados como proxy de la demanda proyectada que generaría el sistema.
3. **Cobertura del catálogo:** Qué proporción de los destinos disponibles llega efectivamente a ser recomendada (problema de cold-start para destinos nicho).
4. **Satisfacción por tipo de preferencia:** Tasa de coincidencia entre el tipo de destino recomendado y la preferencia declarada del usuario.

In [ ]:
from collections import Counter

# Generar recomendaciones para todos los usuarios
print('Generando recomendaciones para todos los usuarios...')
all_recs_data = {uid: recomendar(uid, k=5) for uid in matrix.index}

# 1. Hit Rate comparison (ya calculado)
# 2. Demanda proyectada
demand = Counter()
for uid, recs in all_recs_data.items():
    if not recs.empty:
        demand.update(recs['Destino'].tolist())
top_demand = pd.Series(dict(demand.most_common(10)))

# 3. Cobertura
all_recommended_names = set()
for recs in all_recs_data.values():
    if not recs.empty:
        all_recommended_names.update(recs['Destino'].tolist())
coverage = len(all_recommended_names) / len(destinations)

# 4. Satisfacción por tipo
sat_by_type = {}
for uid, recs in all_recs_data.items():
    if uid not in users_idx.index or recs.empty:
        continue
    prefs = users_idx.loc[uid, 'pref_set']
    match_rate = recs['Tipo'].isin(prefs).mean()
    for p in prefs:
        sat_by_type.setdefault(p, []).append(match_rate)
sat_summary = {t: round(sum(v)/len(v), 3) for t, v in sat_by_type.items()}

# --- Visualizaciones ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Hit Rate por modo
modelos = ['Colaborativo', 'Contenido', 'Híbrido']
hit_rates = [r_cf['Hit Rate'], r_cb['Hit Rate'], r_hy['Hit Rate']]
colores = ['#2f6fff', '#f6a531', '#11a87d']
bars = axes[0].bar(modelos, hit_rates, color=colores)
for bar, val in zip(bars, hit_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.1%}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Usuarios satisfechos por modo\n(Hit Rate Leave-One-Out, K=5)', fontweight='bold')
axes[0].set_ylabel('Hit Rate')
axes[0].set_ylim(0, max(hit_rates) * 1.3)
axes[0].grid(axis='y', alpha=0.25)

# Demanda proyectada
axes[1].barh(top_demand.index[::-1], top_demand.values[::-1], color='#2f6fff', alpha=0.85)
axes[1].set_title('Top 10 destinos más recomendados\n(incremento proyectado en demanda)', fontweight='bold')
axes[1].set_xlabel('Frecuencia en recomendaciones')
axes[1].tick_params(axis='y', labelsize=8)
axes[1].grid(axis='x', alpha=0.25)

# Satisfacción por tipo
sat_keys   = sorted(sat_summary.keys())
sat_values = [sat_summary[k] for k in sat_keys]
type_colors = {'Beach':'#2f6fff','Historical':'#11a87d','Nature':'#f6a531',
               'Adventure':'#e06d2f','City':'#8a5cf5'}
bars3 = axes[2].bar(sat_keys, sat_values,
                    color=[type_colors.get(k,'#aaa') for k in sat_keys])
for bar, val in zip(bars3, sat_values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.1%}', ha='center', fontweight='bold', fontsize=9)
axes[2].set_title('Coincidencia en Top-5 con preferencia\ndeclarada del usuario', fontweight='bold')
axes[2].set_ylabel('Tasa de coincidencia promedio')
axes[2].set_ylim(0, 1.15)
axes[2].grid(axis='y', alpha=0.25)
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle('Análisis de efectividad del sistema de recomendación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Resumen numérico
print(f'\nCobertura del catálogo: {coverage:.1%} ({len(all_recommended_names)}/{len(destinations)} destinos alcanzados)')
print(f'Destinos nunca recomendados (cold-start): {len(destinations) - len(all_recommended_names)}')
print(f'\nSatisfacción promedio por tipo de preferencia:')
for t, v in sorted(sat_summary.items()):
    print(f'  {t:12s}: {v:.1%}')

## 9. Conclusiones

### Resultados obtenidos

| Modo | Precision@5 | Recall@5 | NDCG@5 | Hit Rate |
|------|------------|---------|--------|----------|
| Colaborativo | 0.1580 | 0.7901 | 0.6993 | 79.0% |
| Contenido | 0.0011 | 0.0055 | 0.0035 | 0.6% |
| **Híbrido** | **0.1414** | **0.7072** | **0.6258** | **70.7%** |

### Interpretación

**Filtrado colaborativo (CF)** es el componente dominante. Con un Hit Rate del 79% en Leave-One-Out (K=5), el sistema recupera correctamente el destino oculto para 4 de cada 5 usuarios evaluados. Esto refleja que el dataset tiene patrones de preferencia colectivos claros y suficiente densidad de interacciones.

**Filtrado por contenido solo** tiene un rendimiento muy bajo (0.6%). La razón: las preferencias declaradas como "Beaches" abarcan centenares de destinos tipo `Beach`, por lo que la probabilidad de que el item oculto específico quede en Top-5 es mínima. El contenido aporta valor como señal secundaria, no como señal principal.

**Sistema híbrido** baja ligeramente respecto al CF puro (70.7% vs 79%) porque el componente de contenido introduce algo de ruido al diluir el score CF. Sin embargo, el híbrido es más robusto para usuarios con pocos datos históricos (cold-start parcial), donde el CF puro no tiene suficiente señal.

### Efectividad operacional

- **84–86% de los Top-5** recomendados coinciden con el tipo de preferencia declarada por el usuario.
- **72% del catálogo** de destinos es alcanzado por el sistema. El 28% restante corresponde a destinos poco visitados (cold-start), que requerirían estrategias de diversificación (exploración aleatoria o boost de popularidad).
- Los destinos con mayor demanda proyectada son: Taj Mahal, Kerala Backwaters, Goa Beaches, Jaipur City y Leh Ladakh — datos accionables para que la empresa priorice rutas y capacidad.

### Mejoras potenciales

1. **Modelo latente (SVD / ALS):** descomponer la matriz usuario-destino en factores latentes para mayor precisión en datasets esparsos.
2. **Diversificación:** agregar penalización por categoría ya recomendada para evitar listas homogéneas.
3. **Contexto temporal:** ponderar más las interacciones recientes usando `VisitDate` del historial.
4. **Cold-start de usuarios nuevos:** onboarding con preferencias declaradas y boost de popularidad hasta acumular historial suficiente.